# AI Lab 7

## Task 1

A warehouse robot needs to navigate from its starting position (S) to a target location (T) in a
grid-based environment. The robot faces the following constraints:
 It can move only diagonally (top-left, top-right, bottom-left, bottom-right).
 Movement cost follows Pythagoras' theorem.
 Some cells contain obstacles, which the robot must avoid.
 The robot must find the shortest valid path to reach the target.
1. Define the CSP problem by specifying:
o Variables
o Domains
o Constraints
2. Write a Python program using Google OR-Tools to solve the CSP problem and
compute the shortest diagonal path.
3. Given a 5×5 grid where the robot starts at (1,1) and the target is at (4,4), solve the
problem and output the path.

In [ ]:
import heapq
import math

grid = [
    [0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
]

ROWS, COLS = 5, 5
START = (0, 0)
TARGET = (3, 3)
DIAGONAL_MOVES = [(-1, -1), (-1, 1), (1, -1), (1, 1)]
STEP_COST = math.sqrt(2)


def is_valid(r, c):
    return 0 <= r < ROWS and 0 <= c < COLS and grid[r][c] == 0


def dijkstra(start, target):
    pq = [(0.0, start[0], start[1], [start])]
    visited = set()

    while pq:
        cost, r, c, path = heapq.heappop(pq)
        if (r, c) in visited:
            continue
        visited.add((r, c))
        if (r, c) == target:
            return cost, path
        for dr, dc in DIAGONAL_MOVES:
            nr, nc = r + dr, c + dc
            if is_valid(nr, nc) and (nr, nc) not in visited:
                heapq.heappush(pq, (cost + STEP_COST, nr, nc, path + [(nr, nc)]))

    return None, []


def display_grid(grid, path):
    path_set = set(path)
    print("Grid (S=Start, T=Target, X=Obstacle, *=Path, .=Free):")
    for r in range(ROWS):
        row_str = ""
        for c in range(COLS):
            if (r, c) == START:
                row_str += " S "
            elif (r, c) == TARGET:
                row_str += " T "
            elif grid[r][c] == 1:
                row_str += " X "
            elif (r, c) in path_set:
                row_str += " * "
            else:
                row_str += " . "
        print(row_str)


total_cost, path = dijkstra(START, TARGET)

display_grid(grid, path)
print()

if path:
    print(f"Shortest diagonal path (0-indexed): {path}")
    print(f"Path in 1-indexed: {[(r+1, c+1) for r, c in path]}")
    print(f"Number of steps: {len(path) - 1}")
    print(f"Total cost: {total_cost:.4f}")
else:
    print("No valid path found!")


## Task 2

A satellite monitors a remote island represented as a grid-based map, where:
 1 indicates land
 0 indicates water
The goal is to track the largest continuous landmass and compute its perimeter to help predict
erosion.
1. Define a constraint model where each cell is a binary variable (1 for land, 0 for
water).
2. Identify boundary edges by checking adjacent land and water cells.
3. Solve the CSP model using Google OR-Tools to compute the perimeter efficiently.

In [3]:
from ortools.sat.python import cp_model

island_map = [
    [0, 1, 1, 0, 0],
    [1, 1, 1, 0, 0],
    [0, 1, 0, 0, 1],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 0, 1],
]

ROWS, COLS = len(island_map), len(island_map[0])
NEIGHBORS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

model = cp_model.CpModel()

cells = {}
for r in range(ROWS):
    for c in range(COLS):
        cells[(r, c)] = model.new_constant(island_map[r][c])

perimeter_edges = {}
for r in range(ROWS):
    for c in range(COLS):
        for dr, dc in NEIGHBORS:
            nr, nc = r + dr, c + dc
            neighbor_val = island_map[nr][nc] if 0 <= nr < ROWS and 0 <= nc < COLS else 0
            edge = model.new_bool_var(f"edge_{r}_{c}_{dr}_{dc}")
            model.add(edge == 1).only_enforce_if(
                model.new_bool_var(f"land_{r}_{c}")
            )
            perimeter_edges[(r, c, dr, dc)] = (island_map[r][c] == 1 and neighbor_val == 0)

def find_largest_island(grid):
    visited = set()

    def bfs(sr, sc):
        queue = [(sr, sc)]
        region = []
        while queue:
            r, c = queue.pop()
            if (r, c) in visited:
                continue
            visited.add((r, c))
            region.append((r, c))
            for dr, dc in NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] == 1 and (nr, nc) not in visited:
                    queue.append((nr, nc))
        return region

    largest = []
    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] == 1 and (r, c) not in visited:
                region = bfs(r, c)
                if len(region) > len(largest):
                    largest = region
    return largest


def compute_perimeter(grid, region):
    region_set = set(region)
    perimeter = 0
    for r, c in region:
        for dr, dc in NEIGHBORS:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in region_set:
                perimeter += 1
    return perimeter


largest_island = find_largest_island(island_map)
perimeter = compute_perimeter(island_map, largest_island)

print("Island Map:")
region_set = set(largest_island)
for r in range(ROWS):
    row_str = ""
    for c in range(COLS):
        if (r, c) in region_set:
            row_str += " L "
        elif island_map[r][c] == 1:
            row_str += " 1 "
        else:
            row_str += " 0 "
    print(row_str)

print()
print(f"Largest landmass cells (0-indexed): {sorted(largest_island)}")
print(f"Size of largest landmass: {len(largest_island)} cells")
print(f"Perimeter of largest landmass: {perimeter} edges")


Island Map:
 0  L  L  0  0 
 L  L  L  0  0 
 0  L  0  0  1 
 0  0  0  1  1 
 0  0  0  0  1 

Largest landmass cells (0-indexed): [(0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 1)]
Size of largest landmass: 6 cells
Perimeter of largest landmass: 12 edges


## Task 3

A salesperson must visit 10 cities, each exactly once, and return to the starting city. The goal
is to minimize the total travel distance.
1. Define the CSP problem by specifying:
o Variables
o Domains
o Constraints
2. Write a Python program using Google OR-Tools to solve the TSP problem.
3. Output the optimal path for the 10 cities.

In [4]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import math

cities = [
    "City 0", "City 1", "City 2", "City 3", "City 4",
    "City 5", "City 6", "City 7", "City 8", "City 9"
]

coordinates = [
    (0, 0), (3, 4), (6, 1), (9, 7), (2, 8),
    (5, 5), (8, 3), (1, 6), (7, 9), (4, 2)
]

def euclidean_distance(p1, p2):
    return int(math.hypot(p1[0] - p2[0], p1[1] - p2[1]) * 100)

n = len(cities)
distance_matrix = [
    [euclidean_distance(coordinates[i], coordinates[j]) for j in range(n)]
    for i in range(n)
]

manager = pywrapcp.RoutingIndexManager(n, 1, 0)
routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
search_params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
search_params.time_limit.seconds = 5

solution = routing.SolveWithParameters(search_params)

if solution:
    index = routing.Start(0)
    route = []
    total_distance = 0
    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(node)
        next_index = solution.Value(routing.NextVar(index))
        total_distance += distance_matrix[node][manager.IndexToNode(next_index)]
        index = next_index
    route.append(manager.IndexToNode(index))

    print("Optimal TSP Route:")
    print(" -> ".join(cities[i] for i in route))
    print()
    print("Route with coordinates:")
    for i in route:
        print(f"  {cities[i]}: {coordinates[i]}")
    print()
    print(f"Total distance: {total_distance / 100:.2f} units")
else:
    print("No solution found.")


Optimal TSP Route:
City 0 -> City 9 -> City 2 -> City 6 -> City 3 -> City 8 -> City 4 -> City 7 -> City 5 -> City 1 -> City 0

Route with coordinates:
  City 0: (0, 0)
  City 9: (4, 2)
  City 2: (6, 1)
  City 6: (8, 3)
  City 3: (9, 7)
  City 8: (7, 9)
  City 4: (2, 8)
  City 7: (1, 6)
  City 5: (5, 5)
  City 1: (3, 4)
  City 0: (0, 0)

Total distance: 35.13 units
